# Лекција 09 - Метакогнитивни образац дизајна


## Постављање

Овај ноутбук демонстрира образац дизајна метакогниције користећи Microsoft Agent Framework.

**Преуслови:**
- Деплојмент Azure OpenAI-а подешен помоћу променљивих окружења
- Azure CLI аутентификован (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Шта је метакогниција?

Метакогниција је **размишљање о размишљању**. У контексту агената вештачке интелигенције, то значи правити агенте који могу:

- **Самоиспитивање** сопствених излаза и процеса расуђивања
- **Откривати грешке** и опорављати се на пристојан начин уместо да неуспех прође непримећено
- **Проценити** да ли су њихови одговори потпуни и корисни
- **Прилагодити** своју стратегију када почетни приступ не функционише (нпр., враћање на резервни систем)

Метакогнитивни агент не само да одговара на питања — прати своје перформансе и прилагођава се у покрету.


## Основни и резервни алати

Чест метакогнитивни образац је **стратегија резервног решења**. Агент прво покушава основни алат; ако он не успе (нпр. 404 грешка), агент препознаје неуспех и транспарентно прелази на резервни алат.

То одражава системе из стварног света у којима основне услуге могу бити недоступне и агенти морају сами да дијагностикују проблем пре него што одаберу алтернативни пут.

Испод дефинишемо два алата за претрагу летова:
- **Основни** — покрива Париз, Токио, и Барселону
- **Резервни** — покрива Берлин, Сиднеј, и Њујорк


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Агент који саморазмишља са опоравком од грешака

Доле наведени агент је упућен да прво покуша са примарним системом лета, препозна неуспехе и транспарентно се пребаци на резервни систем. Након сваког одговора кратко самооцени да ли је у потпуности одговорио на питање корисника.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Образац самопроцене

Још једна страна метакогниције је **самопроцена**: посебан агент (или исти агент у другом пролазу) прегледа одговор ради потпуности, тачности и корисности.

Испод креирамо `ResponseEvaluator` агента који оцењује одговоре агента за путовања по три димензије.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Резиме

У овој лекцији сте научили како да изградите **метакогнитивне агенте** користећи Microsoft Agent Framework:

- **Саморефлексија**: Агенти који прате сопствено размишљање и транспарентно саопштавају шта се десило.
- **Опоравак од грешака уз резервне опције**: Шаблон са примарним и резервним алатом где агент детектује неуспехе (нпр. 404 грешке) и аутоматски покушава алтернативни извор.
- **Самопроцена**: Посебан агент-оцењивач који оцењује одговоре по потпуности, тачности и корисности.

Ови обрасци чине агенте робуснијим, транспарентнијим и поузданијим — кључним особинама за постављање у производно окружење.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Одрицање одговорности:
Овај документ је преведен помоћу сервиса за превођење заснованог на вештачкој интелигенци [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да превод буде тачан, имајте у виду да аутоматизовани преводи могу садржати грешке или нетачности. Изворни документ на његовом оригиналном језику треба сматрати ауторитетом. За критичне информације препоручује се професионални превод који обави људски преводилац. Не сносимо одговорност за било каква непоразумевања или погрешна тумачења која могу проистећи из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
